In [205]:
# Install (if needed)
!pip install --upgrade torch

In [206]:
# Imports
import torch
import torch.nn as nn
import time


In [207]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [208]:
# RMSNorm Implementation

class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-8):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        # Efficient RMS computation
        rms = x.norm(dim=-1, keepdim=True)
        return self.scale * (x / (rms * (x.shape[-1] ** -0.5) + self.eps))


In [209]:
# Model Setup

batch_size = 256
seq_len = 1024
d_model = 2048

# Models
layer_norm = nn.LayerNorm(d_model).to(device)
rms_norm = RMSNorm(d_model).to(device)

# Compile (important)
layer_norm = torch.compile(layer_norm)
rms_norm = torch.compile(rms_norm)

# Eval mode
layer_norm.eval()
rms_norm.eval()

OptimizedModule(
  (_orig_mod): RMSNorm()
)

In [210]:
# Dummy Input

x = torch.randn(batch_size, seq_len, d_model, device=device)

In [211]:
layer_norm = torch.compile(nn.LayerNorm(d_model).to(device))
rms_norm = torch.compile(RMSNorm(d_model).to(device))

In [212]:
# Warmup

for _ in range(100):
    with torch.no_grad():
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            _ = layer_norm(x)
            _ = rms_norm(x)

In [213]:
# Benchmark Function

def benchmark(model, x, steps=1000):
    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.time()

    with torch.no_grad():
        for _ in range(steps):
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                _ = model(x)

    if device.type == "cuda":
        torch.cuda.synchronize()

    end = time.time()
    return end - start

In [215]:
# Run Benchmark

ln_time = benchmark(layer_norm, x)
rms_time = benchmark(rms_norm, x)

print(f"\nLayerNorm Time: {ln_time:.4f} sec")
print(f"RMSNorm Time:  {rms_time:.4f} sec")


LayerNorm Time: 19.7979 sec
RMSNorm Time:  17.6696 sec


In [216]:
# Result

if rms_time < ln_time:
    print("RMSNorm is faster")
elif rms_time > ln_time:
    print("LayerNorm is still faster")
else:
    print("Both are nearly identical")

RMSNorm is faster
